# Visualización de Grafos - Google Colab

Este notebook es compatible con Google Colab y permite visualizar grafos en 2D y 3D.

## Instrucciones:
1. En Colab, ejecuta las celdas en orden
2. Si usas archivos en Google Drive, monta la carpeta donde tienes los datos
3. Carga tus archivos CSV (nodes.csv y edges.csv)
4. Personaliza colores y visualización

## 1. Instalar dependencias

In [ ]:
!pip install -q networkx plotly pandas scipy ipywidgets

## 2. Montar Google Drive (opcional - solo si usas Colab)

In [ ]:
from google.colab import drive
import os

try:
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Google Drive montado exitosamente")
except:
    IN_COLAB = False
    print("No estás en Colab o Drive ya está montado")

## 3. Importar librerías

In [ ]:
import pandas as pd
import networkx as nx
import plotly.graph_objects as go
import numpy as np
from ipywidgets import widgets, interact, fixed
import warnings
warnings.filterwarnings('ignore')

## 4. Configurar rutas de datos

In [ ]:
# Para Colab: cambia esta ruta a tu carpeta de datos en Drive
# Ejemplo: '/content/drive/My Drive/graphs/data'
# Para local: usa la ruta a tu carpeta data local

if IN_COLAB:
    DATA_DIR = '/content/drive/MyDrive/graphs/data'  # Cambiar según tu estructura
else:
    DATA_DIR = './data'  # Ruta local

print(f"Buscando grafos en: {DATA_DIR}")
print(f"Existe: {os.path.exists(DATA_DIR)}")

## 5. Cargar lista de grafos disponibles

In [ ]:
def get_available_graphs(data_dir):
    """Obtener lista de grafos válidos en el directorio."""
    available_graphs = []
    
    if not os.path.exists(data_dir):
        print(f"Directorio no encontrado: {data_dir}")
        return []
    
    for item in os.listdir(data_dir):
        item_path = os.path.join(data_dir, item)
        if os.path.isdir(item_path):
            nodes_path = os.path.join(item_path, "nodes.csv")
            edges_path = os.path.join(item_path, "edges.csv")
            if os.path.exists(nodes_path) and os.path.exists(edges_path):
                available_graphs.append(item)
    
    return sorted(available_graphs)

available_graphs = get_available_graphs(DATA_DIR)
print(f"Grafos encontrados: {available_graphs}")

## 6. Funciones para cargar y procesar datos

In [ ]:
def load_graph_data(data_dir, graph_name):
    """Cargar nodes.csv y edges.csv de un grafo."""
    nodes_path = os.path.join(data_dir, graph_name, "nodes.csv")
    edges_path = os.path.join(data_dir, graph_name, "edges.csv")
    
    try:
        nodes_df = pd.read_csv(nodes_path, sep=';')
        edges_df = pd.read_csv(edges_path, sep=';')
        return nodes_df, edges_df
    except Exception as e:
        print(f"Error cargando {graph_name}: {e}")
        return None, None

def create_graph(nodes_df, edges_df):
    """Crear grafo NetworkX."""
    G = nx.DiGraph()
    
    for _, row in nodes_df.iterrows():
        node_id = row.get('id')
        node_label = row.get('label')
        if pd.notna(node_id) and pd.notna(node_label):
            G.add_node(
                str(node_id),
                label=str(node_label),
                **{col: str(row.get(col, '')) for col in nodes_df.columns if col not in ['id', 'label']}
            )
    
    for _, row in edges_df.iterrows():
        source = row.get('src') or row.get('source')
        target = row.get('dst') or row.get('target')
        if pd.notna(source) and pd.notna(target):
            G.add_edge(str(source), str(target))
    
    return G

print("Funciones cargadas correctamente")

## 7. Funciones de visualización

In [ ]:
def generate_color_map(unique_types):
    """Generar mapa de colores automático."""
    color_palette = [
        "#e74c3c", "#3498db", "#2ecc71", "#ff6b6b", "#4dabf7", "#51cf66", "#fab005",
        "#fd7e14", "#6f42c1", "#20c997", "#dc3545", "#ffc107", "#17a2b8", "#28a745",
        "#007bff", "#6c757d", "#343a40", "#f8f9fa"
    ]
    return {type_: color_palette[i % len(color_palette)] for i, type_ in enumerate(unique_types)}

def visualize_2d(G, color_map, title="Grafo 2D"):
    """Visualizar grafo en 2D con Plotly."""
    pos = nx.spring_layout(G, k=2, iterations=50, seed=42)
    
    edge_x, edge_y = [], []
    for source, target in G.edges():
        edge_x.extend([pos[source][0], pos[target][0], None])
        edge_y.extend([pos[source][1], pos[target][1], None])
    
    node_x = [pos[node][0] for node in G.nodes()]
    node_y = [pos[node][1] for node in G.nodes()]
    node_labels = [str(G.nodes[node].get('label', node)) for node in G.nodes()]
    node_types = [str(G.nodes[node].get('type', 'unknown')) for node in G.nodes()]
    node_colors = [color_map.get(t, '#808080') for t in node_types]
    
    fig = go.Figure(data=[
        go.Scatter(x=edge_x, y=edge_y, mode='lines', line=dict(width=0.5, color='#888'), hoverinfo='none'),
        go.Scatter(x=node_x, y=node_y, mode='markers+text', text=node_labels, textposition='top center',
                  marker=dict(size=10, color=node_colors, line=dict(width=2, color='white')), hovertext=node_labels)
    ])
    fig.update_layout(title=title, hovermode='closest', height=700, showlegend=False)
    return fig

def visualize_3d(G, color_map, title="Grafo 3D"):
    """Visualizar grafo en 3D con Plotly."""
    pos_2d = nx.spring_layout(G, k=2, iterations=50, seed=42)
    np.random.seed(42)
    
    pos_3d = {node: [pos_2d[node][0], pos_2d[node][1], np.random.uniform(-1, 1)] for node in G.nodes()}
    
    edge_x, edge_y, edge_z = [], [], []
    for source, target in G.edges():
        edge_x.extend([pos_3d[source][0], pos_3d[target][0], None])
        edge_y.extend([pos_3d[source][1], pos_3d[target][1], None])
        edge_z.extend([pos_3d[source][2], pos_3d[target][2], None])
    
    node_x = [pos_3d[node][0] for node in G.nodes()]
    node_y = [pos_3d[node][1] for node in G.nodes()]
    node_z = [pos_3d[node][2] for node in G.nodes()]
    node_labels = [str(G.nodes[node].get('label', node)) for node in G.nodes()]
    node_types = [str(G.nodes[node].get('type', 'unknown')) for node in G.nodes()]
    node_colors = [color_map.get(t, '#808080') for t in node_types]
    
    fig = go.Figure(data=[
        go.Scatter3d(x=edge_x, y=edge_y, z=edge_z, mode='lines', line=dict(color='rgba(125,125,125,0.2)', width=1), hoverinfo='none'),
        go.Scatter3d(x=node_x, y=node_y, z=node_z, mode='markers+text', text=node_labels, textposition='top center',
                    marker=dict(size=8, color=node_colors, line=dict(width=1, color='white')), hovertext=node_labels)
    ])
    fig.update_layout(title=title, height=700, scene=dict(xaxis_showgrid=False, yaxis_showgrid=False, zaxis_showgrid=False))
    return fig

print("Funciones de visualización cargadas")

## 8. Interfaz interactiva

In [ ]:
if available_graphs:
    # Crear widgets
    graph_dropdown = widgets.Dropdown(
        options=available_graphs,
        description='Seleccionar grafo:',
        style={'description_width': '120px'}
    )
    
    viz_radio = widgets.RadioButtons(
        options=['2D', '3D'],
        description='Visualización:',
        style={'description_width': '120px'}
    )
    
    # Función para actualizar visualización
    def update_visualization(graph_name, viz_type):
        print(f"Cargando {graph_name}...")
        nodes_df, edges_df = load_graph_data(DATA_DIR, graph_name)
        
        if nodes_df is None:
            print("Error cargando datos")
            return
        
        G = create_graph(nodes_df, edges_df)
        print(f"Grafo cargado: {G.number_of_nodes()} nodos, {G.number_of_edges()} aristas")
        
        if 'type' in nodes_df.columns:
            unique_types = sorted(nodes_df['type'].dropna().unique())
            color_map = generate_color_map(unique_types)
        else:
            color_map = {'unknown': '#3498db'}
        
        if viz_type == '2D':
            fig = visualize_2d(G, color_map)
        else:
            fig = visualize_3d(G, color_map)
        
        fig.show()
    
    # Mostrar widgets
    display(widgets.VBox([graph_dropdown, viz_radio]))
    
    # Vincular actualización
    graph_dropdown.observe(lambda change: update_visualization(change.new, viz_radio.value), names='value')
    viz_radio.observe(lambda change: update_visualization(graph_dropdown.value, change.new), names='value')
    
    # Mostrar visualización inicial
    update_visualization(available_graphs[0], '2D')
else:
    print("⚠️ No se encontraron grafos. Verifica la ruta de datos.")

## 9. Estadísticas del grafo

In [ ]:
def show_graph_stats(G):
    """Mostrar estadísticas del grafo."""
    print(f"\n📊 Estadísticas del grafo:")
    print(f"  • Nodos: {G.number_of_nodes()}")
    print(f"  • Aristas: {G.number_of_edges()}")
    print(f"  • Densidad: {nx.density(G):.4f}")
    print(f"  • Número de componentes: {nx.number_weakly_connected_components(G)}")
    
if available_graphs:
    nodes_df, edges_df = load_graph_data(DATA_DIR, available_graphs[0])
    G = create_graph(nodes_df, edges_df)
    show_graph_stats(G)